# STgram-MFN from scratch — every block + training/eval pipeline

A single notebook that builds the entire system explicitly, block by block,
instead of importing the repo's `src/`:

- **B1** log-mel frontend (the "Sgram" branch) and clip extractor
- **B2** `TgramNet` (learned time-domain branch)
- **B3** `MobileFaceNet` backbone (`Bottleneck`, `ConvBlock`)
- **B4** `ArcFace` additive-angular-margin head
- **B5** training loss + anomaly score
- **B6** `STgramMFN` fusion model
- **C** dataset layout helpers, waveform loading, `Dataset`/`DataLoader`
- **D** evaluation: AUC / pAUC / mAUC
- **E** checkpointing, LR schedule, one-epoch loop, full training loop
- **F** run it

```
x_wav --> TgramNet ---------------------------.
                                               v
x_mel (log-mel) -----------------------> concat (B, 2, n_mels, T)
                                               v
                                        MobileFaceNet -> feature (B, 128)
                                               v
                                 (train) ArcFace -> logits -> CE
                                 (test)  -log_softmax -> anomaly score
```

Provenance: the DCASE 2022 Task 2 Top-1 system, Liu et al.
("Anomalous Sound Detection using Spectral-Temporal Information Fusion",
arXiv:2201.05510), reference code `github.com/liuyoude/STgram-MFN`. The code
below is a faithful re-expression of that reference tuned to match this
project's `src/`, so the numbers are directly comparable.

**Before running:** set the runtime to a GPU (`Runtime -> Change runtime type
-> T4 GPU`) and add an `HF_TOKEN` secret (Secrets panel, left sidebar) that can
read `LakoreAI/stgram-mfn-dcase2020-{dev,eval}` and write your checkpoint repo.


## 0 · Environment

### 0.1 · Check the GPU

In [ ]:
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    !nvidia-smi --query-gpu=name,memory.total --format=csv
else:
    print("WARNING: no GPU detected - switch Runtime to a T4 before training.")


### 0.2 · Install dependencies

In [ ]:
import importlib.util
import subprocess
import sys

try:
    import torch

    cuda = torch.cuda.is_available()
except ImportError:
    cuda = False

deps = [
    "numpy>=1.26",
    "scikit-learn>=1.4",
    "soundfile>=0.13.1",
    "huggingface_hub>=0.24",
    "hf_transfer",
    "tqdm",
]
# Keep Colab's CUDA-matched torch; only install torch when it is missing/CPU.
if not cuda:
    deps += ["torch", "torchaudio"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + deps, check=True)

import torch
import torchaudio

print("torch:", torch.__version__, "| torchaudio:", torchaudio.__version__, "| cuda:", torch.cuda.is_available())


### 0.3 · Hugging Face token

In [ ]:
import os

try:
    from google.colab import userdata

    def _secret(name):
        try:
            return userdata.get(name)
        except Exception:
            return None
except ImportError:

    def _secret(name):
        return os.environ.get(name)


HF_TOKEN = _secret("HF_TOKEN")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HF_API_KEY"] = HF_TOKEN
print("HF token:", "set" if HF_TOKEN else "NOT set (data pull and checkpoint push will fail)")


### 0.4 · Configuration

Two dataclasses: `ModelConfig` is architecture only (layers, audio frontend,
ArcFace geometry); `TrainConfig` is the optimization/data budget. `num_classes`
is filled in later from the data, since it equals the number of machine-id
classes.

In [ ]:
from dataclasses import dataclass, field
from typing import Tuple

# DCASE 2022 Task 2 Top-1 bottleneck setting (t, c, n, s) from the reference.
DEFAULT_BOTTLENECK_SETTING = ((2, 128, 2, 2), (4, 128, 2, 2), (4, 128, 2, 2))


@dataclass
class ModelConfig:
    num_classes: int = 0
    embed_dim: int = 128

    c_dim: int = 128
    win_len: int = 1024
    hop_len: int = 512
    tgram_num_layers: int = 3
    n_frames: int = 313
    spatial_size: Tuple[int, int] = (8, 20)
    bottleneck_setting: Tuple[Tuple[int, int, int, int], ...] = field(
        default_factory=lambda: DEFAULT_BOTTLENECK_SETTING
    )

    use_arcface: bool = True
    arcface_m: float = 0.7
    arcface_s: float = 30.0
    arcface_sub: int = 1
    arcface_easy_margin: bool = False

    sample_rate: int = 16000
    n_fft: int = 1024
    n_mels: int = 128
    win_length: int = 1024
    hop_length: int = 512
    power: float = 2.0
    secs: float = 10.0

    def __post_init__(self):
        if self.n_mels != self.c_dim:
            raise ValueError("n_mels must equal c_dim (branches concat channel-wise)")
        if self.win_length != self.win_len or self.hop_length != self.hop_len:
            raise ValueError("audio win/hop must match TgramNet win/hop (shared time axis)")


@dataclass
class TrainConfig:
    machines: list = field(
        default_factory=lambda: ["fan", "pump", "slider", "valve", "ToyCar", "ToyConveyor"]
    )
    data_root: str = "/content/stgram_mfn/raw"
    add_root: str = "/content/stgram_mfn/raw_eval"
    train_subdir: str = "train"
    test_subdir: str = "test"
    ckpt_dir: str = "/content/stgram_mfn/checkpoints"
    run_name: str = "stgram_mfn_scratch"

    epochs: int = 60
    batch_size: int = 64
    accum_steps: int = 2
    num_workers: int = 8
    amp: bool = False
    lr: float = 1e-4
    weight_decay: float = 0.0
    log_every: int = 20
    eval_every: int = 10
    ckpt_every: int = 20
    seed: int = 42

    sched_t_max: int = 17000
    sched_eta_min: float = 1e-5
    sched_start_epoch: int = 5

    save_best: bool = True
    eval_batch_size: int = 32
    hf_push_repo: str = "LakoreAI/stgram-mfn-scratch"


TRAIN_CFG = TrainConfig()
print(TRAIN_CFG)


### 0.5 · Device, seed, and run directory

In [ ]:
import random
from pathlib import Path

import numpy as np
import torch


def detect_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


DEVICE = detect_device()
random.seed(TRAIN_CFG.seed)
np.random.seed(TRAIN_CFG.seed)
torch.manual_seed(TRAIN_CFG.seed)
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(TRAIN_CFG.seed)

DATA_ROOT = Path(TRAIN_CFG.data_root)
ADD_ROOT = Path(TRAIN_CFG.add_root)
RUN_DIR = Path(TRAIN_CFG.ckpt_dir) / TRAIN_CFG.run_name
RUN_DIR.mkdir(parents=True, exist_ok=True)

print("device:", DEVICE)
print("run dir:", RUN_DIR)


## B · Model blocks

### B1 · Log-mel frontend (Sgram) + clip extractor

The Sgram branch is a power mel-spectrogram in dB, unnormalized. A clip is
cropped/padded to exactly `secs` seconds so every sample has the same
`(n_mels, n_frames)` shape — this is what lets TgramNet use a fixed-width
LayerNorm and lets evaluation batch the forward pass.

In [ ]:
import torch
import torch.nn as nn
import torchaudio.transforms as T


def frames_for_seconds(secs, sample_rate, hop_length):
    # 1 + floor(secs * sr / hop) with center=True padding.
    return 1 + int(secs * sample_rate) // hop_length


class Wave2Mel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.mel = T.MelSpectrogram(
            sample_rate=cfg.sample_rate,
            n_fft=cfg.n_fft,
            win_length=cfg.win_length,
            hop_length=cfg.hop_length,
            n_mels=cfg.n_mels,
            power=cfg.power,
        )
        self.to_db = T.AmplitudeToDB(stype="power")

    def forward(self, waveform):
        # (T,) or (B, T) -> (n_mels, n_frames) or (B, n_mels, n_frames)
        return self.to_db(self.mel(waveform))


class FeatureExtractor(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.wav2mel = Wave2Mel(cfg)
        self.clip_samples = int(cfg.secs * cfg.sample_rate)

    def forward(self, waveform):
        # waveform: (T,) mono at cfg.sample_rate -> (x_wav (clip_samples,), x_mel (n_mels, n_frames))
        if waveform.dim() > 1:
            waveform = waveform.mean(dim=0)
        n = waveform.shape[0]
        if n < self.clip_samples:
            waveform = torch.nn.functional.pad(waveform, (0, self.clip_samples - n))
        else:
            waveform = waveform[: self.clip_samples]
        return waveform, self.wav2mel(waveform)


print("n_frames for 10 s:", frames_for_seconds(10.0, 16000, 512))


### B2 · TgramNet — learned time-domain branch

A large-kernel strided 1-D conv whose kernel/stride/padding mirror the STFT
window/hop, so its output shares the mel spectrogram's time axis. Three small
conv blocks refine it; `LayerNorm` normalizes over the **time** axis (hence the
explicit `n_frames`).

In [ ]:
class TgramNet(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.conv_extractor = nn.Conv1d(
            1, cfg.c_dim, cfg.win_len, cfg.hop_len, cfg.win_len // 2, bias=False
        )
        self.conv_encoder = nn.Sequential(
            *[
                nn.Sequential(
                    nn.LayerNorm(cfg.n_frames),
                    nn.LeakyReLU(0.2, inplace=True),
                    nn.Conv1d(cfg.c_dim, cfg.c_dim, 3, 1, 1, bias=False),
                )
                for _ in range(cfg.tgram_num_layers)
            ]
        )

    def forward(self, x):
        # x: (B, 1, T) raw waveform -> (B, c_dim, n_frames)
        return self.conv_encoder(self.conv_extractor(x))


### B3 · MobileFaceNet backbone

Inverted-residual (`Bottleneck`) stack with the DCASE Top-1 setting. Input is
the 2-channel STgram tensor (channel 0 = Sgram, channel 1 = Tgram); output is
the 128-d embedding. `linear7` collapses the spatial grid with a depthwise
conv whose kernel equals `spatial_size` (8, 20) for a 128x313 input.

In [ ]:
import math


class Bottleneck(nn.Module):
    # 1x1 expand -> depthwise -> 1x1 project; residual only if shape preserved.
    def __init__(self, inp, oup, stride, expansion):
        super().__init__()
        self.connect = stride == 1 and inp == oup
        self.conv = nn.Sequential(
            nn.Conv2d(inp, inp * expansion, 1, 1, 0, bias=False),
            nn.BatchNorm2d(inp * expansion),
            nn.PReLU(inp * expansion),
            nn.Conv2d(inp * expansion, inp * expansion, 3, stride, 1,
                      groups=inp * expansion, bias=False),
            nn.BatchNorm2d(inp * expansion),
            nn.PReLU(inp * expansion),
            nn.Conv2d(inp * expansion, oup, 1, 1, 0, bias=False),
            nn.BatchNorm2d(oup),
        )

    def forward(self, x):
        if self.connect:
            return x + self.conv(x)
        return self.conv(x)


class ConvBlock(nn.Module):
    def __init__(self, inp, oup, k, s, p, dw=False, linear=False):
        super().__init__()
        self.linear = linear
        if dw:
            self.conv = nn.Conv2d(inp, oup, k, s, p, groups=inp, bias=False)
        else:
            self.conv = nn.Conv2d(inp, oup, k, s, p, bias=False)
        self.bn = nn.BatchNorm2d(oup)
        if not linear:
            self.prelu = nn.PReLU(oup)

    def forward(self, x):
        x = self.bn(self.conv(x))
        if self.linear:
            return x
        return self.prelu(x)


class MobileFaceNet(nn.Module):
    def __init__(self, cfg, in_channels=2):
        super().__init__()
        self.cfg = cfg
        setting = cfg.bottleneck_setting

        self.conv1 = ConvBlock(in_channels, 64, 3, 2, 1)
        self.dw_conv1 = ConvBlock(64, 64, 3, 1, 1, dw=True)

        self.inplanes = 64
        self.blocks = self._make_layer(Bottleneck, setting)

        self.conv2 = ConvBlock(setting[-1][1], 512, 1, 1, 0)
        self.linear7 = ConvBlock(512, 512, cfg.spatial_size, 1, 0, dw=True, linear=True)
        self.linear1 = ConvBlock(512, cfg.embed_dim, 1, 1, 0, linear=True)
        self.fc_out = nn.Linear(cfg.embed_dim, cfg.num_classes)

        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                n = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
                m.weight.data.normal_(0, math.sqrt(2.0 / n))
            elif isinstance(m, nn.BatchNorm2d):
                m.weight.data.fill_(1)
                m.bias.data.zero_()

    def _make_layer(self, block, setting):
        layers = []
        for t, c, n, s in setting:
            for i in range(n):
                layers.append(block(self.inplanes, c, s if i == 0 else 1, t))
                self.inplanes = c
        return nn.Sequential(*layers)

    def forward(self, x, label=None):
        # x: (B, 2, n_mels, n_frames) -> (logits (B, num_classes), feature (B, embed_dim))
        x = self.conv1(x)
        x = self.dw_conv1(x)
        x = self.blocks(x)
        x = self.conv2(x)
        x = self.linear7(x)
        x = self.linear1(x)
        feature = x.view(x.size(0), -1)
        out = self.fc_out(feature)
        return out, feature


### B4 · ArcFace head

Additive angular margin on normalized embeddings. During training the logits for
the target class are shrunk by `cos(theta + m)` (scaled by `s`), which tightens
same-id clusters. `sub=1` is plain ArcFace.

In [ ]:
import torch.nn.functional as F
from torch.nn import Parameter


class ArcMarginProduct(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.in_features = cfg.embed_dim
        self.out_features = cfg.num_classes
        self.s = cfg.arcface_s
        self.m = cfg.arcface_m
        self.sub = cfg.arcface_sub
        self.weight = Parameter(torch.Tensor(self.out_features * self.sub, self.in_features))
        nn.init.xavier_uniform_(self.weight)

        self.easy_margin = cfg.arcface_easy_margin
        self.cos_m = math.cos(self.m)
        self.sin_m = math.sin(self.m)
        self.th = math.cos(math.pi - self.m)
        self.mm = math.sin(math.pi - self.m) * self.m

    def forward(self, x, label):
        cosine = F.linear(F.normalize(x), F.normalize(self.weight))
        if self.sub > 1:
            cosine = cosine.view(-1, self.out_features, self.sub)
            cosine, _ = torch.max(cosine, dim=2)
        sine = torch.sqrt((1.0 - torch.pow(cosine, 2)).clamp_min(0))
        phi = cosine * self.cos_m - sine * self.sin_m
        if self.easy_margin:
            phi = torch.where(cosine > 0, phi, cosine)
        else:
            phi = torch.where((cosine - self.th) > 0, phi, cosine - self.mm)

        one_hot = torch.zeros(cosine.size(), device=x.device)
        one_hot.scatter_(1, label.view(-1, 1).long(), 1)
        output = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        return output * self.s


### B5 · Loss and anomaly score

Training is ordinary cross-entropy on the ArcFace logits. At test time there is
no anomaly class: the score is `-log_softmax(logits)[own machine id]`. A normal
recording of `id_XX` is confidently `id_XX` (low score); an anomaly is not
(high score).

In [ ]:
class ASDLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.ce = nn.CrossEntropyLoss()

    def forward(self, logits, labels):
        return self.ce(logits, labels)


@torch.no_grad()
def anomaly_score(logits, labels):
    # -log_softmax(logits)[label]; larger = more anomalous. Returns (B,)
    log_probs = F.log_softmax(logits, dim=1)
    return -log_probs.gather(1, labels.view(-1, 1).long()).squeeze(1)


### B6 · STgram-MFN (fusion)

Both branches are concatenated channel-wise, which is why MobileFaceNet is built
with `in_channels=2`.

In [ ]:
class STgramMFN(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.use_arcface = cfg.use_arcface
        self.arcface = ArcMarginProduct(cfg) if cfg.use_arcface else None
        self.tgramnet = TgramNet(cfg)
        self.mobilefacenet = MobileFaceNet(cfg, in_channels=2)

    def get_tgram(self, x_wav):
        return self.tgramnet(x_wav.unsqueeze(1))

    def forward(self, x_wav, x_mel, label=None):
        if x_wav.dim() == 2:
            x_wav = x_wav.unsqueeze(1)
        if x_mel.dim() == 3:
            x_mel = x_mel.unsqueeze(1)

        x_t = self.tgramnet(x_wav).unsqueeze(1)
        x = torch.cat((x_mel, x_t), dim=1)

        out, feature = self.mobilefacenet(x, label)
        if self.arcface is not None:
            if label is None:
                raise ValueError("ArcFace requires label at forward time")
            out = self.arcface(feature, label)
        return out, feature


### B7 · Sanity check the shapes

In [ ]:
_cfg = ModelConfig(num_classes=6)
_ext = FeatureExtractor(_cfg)
_wave = torch.randn(int(_cfg.secs * _cfg.sample_rate))
_x_wav, _x_mel = _ext(_wave)
print("single clip -> x_wav:", tuple(_x_wav.shape), "| x_mel:", tuple(_x_mel.shape))

_model = STgramMFN(_cfg)
# Batch of 2: BatchNorm needs more than one value per channel at the collapsed (1, 1) grid.
_x_wav_b = torch.stack([_x_wav, _x_wav])
_x_mel_b = torch.stack([_x_mel, _x_mel])
_out, _feat = _model(_x_wav_b, _x_mel_b, torch.tensor([0, 1]))
print("feature:", tuple(_feat.shape), "| logits:", tuple(_out.shape))
print("params:", sum(p.numel() for p in _model.parameters()))


## C · Data pipeline

### C1 · Dataset layout helpers

DCASE ships `<root>/<machine>/<split>/<file>.wav`; filenames encode the machine
id (`normal_id_00_00000000.wav`, `anomaly_id_00_...`). Machine type comes from
the path, machine id from the `id_XX` token. One class label is assigned per
(machine type, machine id) pair — the self-supervised pretext.

In [ ]:
import glob
import os
import re


def get_filename_list(dir_path, pattern="*", ext="*"):
    filename_list = []
    for root, _, _ in os.walk(dir_path):
        filename_list += sorted(glob.glob(os.path.join(root, f"{pattern}.{ext}")))
    return filename_list


def get_machine_id_list(data_dir):
    return sorted(
        set(_id for path in get_filename_list(data_dir) for _id in re.findall(r"id_[0-9][0-9]", path))
    )


def metadata_to_label(data_dirs):
    meta2label, label2meta = {}, {}
    label = 0
    for data_dir in data_dirs:
        machine = os.path.basename(os.path.dirname(os.path.normpath(data_dir)))
        for id_str in get_machine_id_list(data_dir):
            meta = f"{machine}-{id_str}"
            meta2label[meta] = label
            label2meta[label] = meta
            label += 1
    return meta2label, label2meta


def machine_of_file(file_path):
    return file_path.split(os.sep)[-3]


def machine_id_of_file(file_path):
    matches = re.findall(r"id_[0-9][0-9]", file_path)
    if not matches:
        raise ValueError(f"no id_XX token in {file_path!r}")
    return matches[0]


def create_test_file_list(target_dir, id_name, prefix_normal="normal", prefix_anomaly="anomaly", ext="wav"):
    # (files, labels) for one id: normal=0, anomaly=1
    normal_files = sorted(glob.glob(os.path.join(target_dir, f"{prefix_normal}_{id_name}*.{ext}")))
    anomaly_files = sorted(glob.glob(os.path.join(target_dir, f"{prefix_anomaly}_{id_name}*.{ext}")))
    files = np.concatenate((normal_files, anomaly_files), axis=0)
    labels = np.concatenate((np.zeros(len(normal_files)), np.ones(len(anomaly_files))), axis=0)
    return files, labels


def build_train_file_list(train_dirs):
    file_list = []
    for train_dir in train_dirs:
        file_list.extend(get_filename_list(train_dir))
    return file_list


### C2 · Waveform loading + `Dataset`

`soundfile` is used instead of `torchaudio.load` (which now needs TorchCodec),
and `torchaudio.functional.resample` handles any sample-rate mismatch.

In [ ]:
import soundfile as sf
import torchaudio
from torch.utils.data import Dataset


def load_waveform(path, target_sr):
    data, sr = sf.read(str(path), dtype="float32", always_2d=True)
    waveform = torch.from_numpy(data.mean(axis=1))
    if sr != target_sr:
        waveform = torchaudio.functional.resample(waveform, sr, target_sr)
    return waveform


class ASDDataset(Dataset):
    def __init__(self, file_list, meta2label, extractor, load_in_memory=False):
        self.file_list = file_list
        self.meta2label = meta2label
        self.extractor = extractor
        self.load_in_memory = load_in_memory
        self.data_list = [self.transform(f) for f in file_list] if load_in_memory else []

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, item):
        if self.load_in_memory:
            return self.data_list[item]
        return self.transform(self.file_list[item])

    def transform(self, filename):
        machine = machine_of_file(filename)
        id_str = machine_id_of_file(filename)
        label = self.meta2label[f"{machine}-{id_str}"]
        waveform = load_waveform(filename, self.extractor.cfg.sample_rate)
        x_wav, x_mel = self.extractor(waveform)
        return x_wav, x_mel, label


def make_extractor(cfg):
    return FeatureExtractor(cfg)


### C3 · Pull the data

The two DCASE sets live on HF as loose wavs (~54k files), so this is the slow
step (~40-60 min). It is resumable: re-run the cell and completed files are
skipped.

In [ ]:
import pathlib

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
from huggingface_hub import snapshot_download

SETS = [
    (DATA_ROOT, "LakoreAI/stgram-mfn-dcase2020-dev"),
    (ADD_ROOT, "LakoreAI/stgram-mfn-dcase2020-eval"),
]


def complete(root):
    root = pathlib.Path(root)
    return all(
        (root / m / s).is_dir() and any((root / m / s).glob("*.wav"))
        for m in TRAIN_CFG.machines
        for s in ("train", "test")
    )


for local, repo in SETS:
    if complete(local):
        print("[ok]", local, "complete - skipping download")
        continue
    print("pulling", repo, "->", local, "(re-run if it drops)")
    snapshot_download(repo_id=repo, repo_type="dataset", local_dir=str(local), max_workers=16)

for local, _ in SETS:
    print(local, "wavs:", sum(1 for _ in pathlib.Path(local).rglob("*.wav")))


### C4 · Build file lists, labels, and loaders

In [ ]:
from torch.utils.data import DataLoader


def build_train_dirs(cfg):
    dirs = [str(Path(cfg.data_root) / m / cfg.train_subdir) for m in cfg.machines]
    if cfg.add_root:
        dirs += [str(Path(cfg.add_root) / m / cfg.train_subdir) for m in cfg.machines]
    return dirs


def build_test_dirs(cfg):
    return [str(Path(cfg.data_root) / m / cfg.test_subdir) for m in cfg.machines]


train_dirs = build_train_dirs(TRAIN_CFG)
test_dirs = build_test_dirs(TRAIN_CFG)
meta2label, label2meta = metadata_to_label(train_dirs)
NUM_CLASSES = len(meta2label)
print("machine-id classes:", NUM_CLASSES)

model_cfg = ModelConfig(num_classes=NUM_CLASSES)
extractor = FeatureExtractor(model_cfg).to(DEVICE)

train_files = build_train_file_list(train_dirs)
print("train clips:", len(train_files))

train_ds = ASDDataset(train_files, meta2label, extractor, load_in_memory=False)
train_loader = DataLoader(
    train_ds,
    batch_size=TRAIN_CFG.batch_size,
    shuffle=True,
    drop_last=True,
    num_workers=TRAIN_CFG.num_workers,
    pin_memory=(DEVICE.type == "cuda"),
)

x_wav_b, x_mel_b, y_b = next(iter(train_loader))
print("batch:", tuple(x_wav_b.shape), tuple(x_mel_b.shape), tuple(y_b.shape), "| labels:", sorted(set(y_b.tolist()))[:6])


## D · Evaluation pipeline

### D1 · Score files, then AUC / pAUC / mAUC

For each machine id: score every test file by `-log_softmax(logits)[own id]`,
then compute AUC and partial-AUC (max FPR 0.1). per-machine AUC is the mean over
its ids; **mAUC is the minimum** over its ids (the reference's stability metric).
The report averages each per-machine value equally across machine types.

In [ ]:
from sklearn.metrics import roc_auc_score

MAX_FPR = 0.1


@torch.no_grad()
def score_files(model, extractor, file_paths, label, device, batch_size=32):
    model.eval()
    scores = []
    for start in range(0, len(file_paths), batch_size):
        chunk = file_paths[start : start + batch_size]
        wavs, mels = [], []
        for f in chunk:
            waveform = load_waveform(f, extractor.cfg.sample_rate).to(device)
            x_wav, x_mel = extractor(waveform)
            wavs.append(x_wav)
            mels.append(x_mel)
        x_wav = torch.stack(wavs)
        x_mel = torch.stack(mels)
        labels = torch.full((len(chunk),), label, dtype=torch.long, device=device)
        logits, _ = model(x_wav, x_mel, labels)
        scores.extend(anomaly_score(logits, labels).tolist())
    return scores


@torch.no_grad()
def score_file(model, extractor, file_path, label, device):
    return score_files(model, extractor, [file_path], label, device, batch_size=1)[0]


def evaluate(model, cfg, meta2label, test_dirs, device, extractor=None, batch_size=32):
    model.eval()
    extractor = extractor or FeatureExtractor(cfg).to(device)

    per_machine, per_id = {}, {}
    machine_aucs, machine_paucs, machine_maucs = [], [], []

    for target_dir in sorted(test_dirs):
        machine_type = Path(target_dir).parent.name
        aucs, paucs = [], []
        for id_str in get_machine_id_list(target_dir):
            meta = f"{machine_type}-{id_str}"
            label = meta2label[meta]
            test_files, y_true = create_test_file_list(target_dir, id_str)
            if len(np.unique(y_true)) < 2:
                continue
            y_pred = score_files(model, extractor, list(test_files), label, device, batch_size=batch_size)
            auc = roc_auc_score(y_true, y_pred)
            pauc = roc_auc_score(y_true, y_pred, max_fpr=MAX_FPR)
            aucs.append(auc)
            paucs.append(pauc)
            per_id[f"{machine_type}-{id_str}"] = {"auc": auc, "pauc": pauc}
        if aucs:
            per_machine[machine_type] = {
                "auc": float(np.mean(aucs)),
                "pauc": float(np.mean(paucs)),
                "mauc": float(np.min(aucs)),
            }
            machine_aucs.append(float(np.mean(aucs)))
            machine_paucs.append(float(np.mean(paucs)))
            machine_maucs.append(float(np.min(aucs)))

    return {
        "auc": float(np.mean(machine_aucs)) if machine_aucs else float("nan"),
        "pauc": float(np.mean(machine_paucs)) if machine_paucs else float("nan"),
        "mauc": float(np.mean(machine_maucs)) if machine_maucs else float("nan"),
        "per_machine": per_machine,
        "per_id": per_id,
    }


def format_report(result):
    lines = []
    for machine, m in sorted(result["per_machine"].items()):
        lines.append(f"{machine:14s} AUC={m['auc'] * 100:6.3f}  pAUC={m['pauc'] * 100:6.3f}  mAUC={m['mauc'] * 100:6.3f}")
    lines.append(f"{'TOTAL':14s} AUC={result['auc'] * 100:6.3f}  pAUC={result['pauc'] * 100:6.3f}  mAUC={result['mauc'] * 100:6.3f}")
    return "\n".join(lines)


## E · Training pipeline

### E1 · Checkpointing and HF push helper

In [ ]:
import json


def save_checkpoint(model, optimizer, step, path, extra=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(
        {"model": model.state_dict(), "optimizer": optimizer.state_dict(), "step": step, "extra": extra or {}},
        path,
    )


def load_checkpoint(path, model, optimizer=None, map_location=None):
    ckpt = torch.load(path, map_location=map_location)
    model.load_state_dict(ckpt["model"])
    if optimizer is not None and "optimizer" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer"])
    return ckpt


def save_json(data, filename):
    filename = Path(filename)
    filename.parent.mkdir(parents=True, exist_ok=True)
    filename.write_text(json.dumps(data, indent=2))


HF_REPO = TRAIN_CFG.hf_push_repo


def hf_push_file(local_path, run_name):
    # Fail-soft: a push must never kill training.
    if not HF_REPO:
        return
    try:
        from huggingface_hub import HfApi

        api = HfApi()
        api.create_repo(HF_REPO, repo_type="model", exist_ok=True, private=True)
        api.upload_file(
            path_or_fileobj=str(local_path),
            path_in_repo=f"{run_name}/{Path(local_path).name}",
            repo_id=HF_REPO,
            repo_type="model",
        )
        print("  [hf] pushed", Path(local_path).name)
    except Exception as e:
        print("  [hf] push failed:", e)


def hf_latest_epoch(run_name):
    if not HF_REPO:
        return None
    try:
        from huggingface_hub import HfApi

        files = HfApi().list_repo_files(HF_REPO, repo_type="model")
    except Exception as e:
        print("  [hf] list skipped:", e)
        return None
    prefix, suffix = run_name + "/epoch_", ".pt"
    epochs = []
    for f in files:
        if f.startswith(prefix) and f.endswith(suffix):
            try:
                epochs.append(int(f[len(prefix) : -len(suffix)]))
            except ValueError:
                pass
    return max(epochs) if epochs else None


def hf_download(file_in_repo, local_path):
    import shutil

    from huggingface_hub import hf_hub_download

    p = hf_hub_download(repo_id=HF_REPO, filename=file_in_repo, repo_type="model")
    shutil.copy(p, local_path)


### E2 · One training epoch (with gradient accumulation)

In [ ]:
def train_one_epoch(model, criterion, loader, optimizer, scaler, scheduler, step, epoch):
    model.train()
    epoch_losses, pending = [], []
    optimizer.zero_grad()

    for x_wavs, x_mels, labels in loader:
        x_wavs = x_wavs.float().to(DEVICE)
        x_mels = x_mels.float().to(DEVICE)
        labels = labels.reshape(-1).long().to(DEVICE)

        with torch.autocast(device_type=DEVICE.type, enabled=scaler.is_enabled()):
            logits, _ = model(x_wavs, x_mels, labels)
            loss = criterion(logits, labels)

        scaler.scale(loss / TRAIN_CFG.accum_steps).backward()
        pending.append(loss.item())

        if len(pending) == TRAIN_CFG.accum_steps:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            step += 1
            if scheduler is not None and epoch >= TRAIN_CFG.sched_start_epoch:
                scheduler.step()
            avg_loss = sum(pending) / len(pending)
            epoch_losses.append(avg_loss)
            pending = []
            if step % TRAIN_CFG.log_every == 0 or step == 1:
                print(f"  epoch {epoch:3d}  step {step:6d}  loss={avg_loss:.4f}")

    if pending:
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()
        step += 1
        if scheduler is not None and epoch >= TRAIN_CFG.sched_start_epoch:
            scheduler.step()
        epoch_losses.append(sum(pending) / len(pending))

    return (sum(epoch_losses) / len(epoch_losses) if epoch_losses else float("nan")), step


### E3 · Full training loop

Adam + cosine LR (held constant for the first `sched_start_epoch` epochs, then
annealed per optimizer step). Every `eval_every` epochs it validates on the
official test set and saves `best.pt` on the best AUC; every `ckpt_every` it
saves `epoch_N.pt` plus `last.pt`. If `hf_push_repo` is set, checkpoints stream
to HF and the loop auto-resumes from the latest epoch on the next start.

In [ ]:
def train():
    model = STgramMFN(model_cfg).to(DEVICE)
    print("params:", sum(p.numel() for p in model.parameters()))
    criterion = ASDLoss().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=TRAIN_CFG.lr, weight_decay=TRAIN_CFG.weight_decay)
    scaler = torch.amp.GradScaler(
        "cuda" if DEVICE.type == "cuda" else "cpu",
        enabled=(TRAIN_CFG.amp and DEVICE.type == "cuda"),
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=TRAIN_CFG.sched_t_max, eta_min=TRAIN_CFG.sched_eta_min
    )

    start_epoch, step = 1, 0
    best_auc = float("-inf")

    last_path = RUN_DIR / "last.pt"
    if last_path.exists():
        ckpt = load_checkpoint(last_path, model, optimizer, map_location=str(DEVICE))
        step = int(ckpt.get("step", 0))
        start_epoch = int(ckpt.get("extra", {}).get("epoch", 0)) + 1
        print(f"resumed {last_path.name}: epoch {start_epoch}, step {step}")
    elif HF_REPO:
        n = hf_latest_epoch(TRAIN_CFG.run_name)
        if n:
            local = RUN_DIR / f"epoch_{n}.pt"
            if not local.exists():
                hf_download(f"{TRAIN_CFG.run_name}/epoch_{n}.pt", local)
            ckpt = load_checkpoint(local, model, optimizer, map_location=str(DEVICE))
            step = int(ckpt.get("step", 0))
            start_epoch = int(ckpt.get("extra", {}).get("epoch", 0)) + 1
            print(f"auto-resumed from HF epoch {n}: epoch {start_epoch}, step {step}")

    best_path = RUN_DIR / "best.pt"
    if best_path.exists():
        try:
            best_auc = float(torch.load(best_path, map_location="cpu").get("extra", {}).get("auc", float("-inf")))
        except Exception:
            pass

    history = []
    for epoch in range(start_epoch, TRAIN_CFG.epochs + 1):
        train_loss, step = train_one_epoch(model, criterion, train_loader, optimizer, scaler, scheduler, step, epoch)
        record = {"epoch": epoch, "step": step, "loss": train_loss}
        print(f"epoch {epoch:3d}/{TRAIN_CFG.epochs}  train_loss={train_loss:.4f}")

        if epoch % TRAIN_CFG.eval_every == 0 or epoch == TRAIN_CFG.epochs:
            result = evaluate(model, model_cfg, meta2label, test_dirs, DEVICE, extractor, batch_size=TRAIN_CFG.eval_batch_size)
            record.update({"auc": result["auc"], "pauc": result["pauc"], "mauc": result["mauc"]})
            print(format_report(result))
            if TRAIN_CFG.save_best and result["auc"] > best_auc:
                best_auc = result["auc"]
                save_checkpoint(model, optimizer, step, best_path,
                                extra={"epoch": epoch, "auc": result["auc"], "meta2label": meta2label})
                hf_push_file(best_path, TRAIN_CFG.run_name)
        history.append(record)
        save_json(history, RUN_DIR / "train_log.json")

        if epoch % TRAIN_CFG.ckpt_every == 0 or epoch == TRAIN_CFG.epochs:
            save_checkpoint(model, optimizer, step, RUN_DIR / f"epoch_{epoch}.pt",
                            extra={"epoch": epoch, "meta2label": meta2label})
            save_checkpoint(model, optimizer, step, last_path,
                            extra={"epoch": epoch, "meta2label": meta2label})
            print("  saved checkpoint epoch", epoch)
            hf_push_file(RUN_DIR / f"epoch_{epoch}.pt", TRAIN_CFG.run_name)

    save_json(history, RUN_DIR / "train_log.json")
    print("done. best AUC =", best_auc)
    return history


## F · Run it

In [ ]:
history = train()


### F1 · Plot the training history

In [ ]:
import matplotlib.pyplot as plt

epochs = [r["epoch"] for r in history]
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(epochs, [r.get("loss") for r in history], marker=".")
ax[0].set_title("train loss")
ax[0].set_xlabel("epoch")
ax[1].plot(epochs, [r.get("auc") for r in history], marker="o")
ax[1].set_title("test AUC")
ax[1].set_xlabel("epoch")
plt.tight_layout()
plt.show()
best = max((r for r in history if r.get("auc") is not None), key=lambda r: r["auc"], default=None)
print("best epoch:", best)


### F2 · Evaluate the saved best checkpoint

In [ ]:
best_path = RUN_DIR / "best.pt"
if not best_path.exists():
    print("no best.pt yet - run the training cell first")
else:
    best_model = STgramMFN(model_cfg).to(DEVICE)
    load_checkpoint(best_path, best_model, map_location=str(DEVICE))
    result = evaluate(best_model, model_cfg, meta2label, test_dirs, DEVICE, extractor, batch_size=TRAIN_CFG.eval_batch_size)
    print(format_report(result))


---

### Notes

- **Comparable to the repo.** This notebook is a line-for-line re-expression of
  `src/config.py` + `src/modules/*` + `src/data.py` + `src/pipelines/{eval,train}.py`,
  so configs from `configs/*.yaml` map 1:1 onto `TrainConfig` here. Reference
  target (300 epochs): **AUC 92.36, mAUC 84.86**.
- **Resilience.** With `hf_push_repo` set, a dropped runtime costs at most one
  eval interval: re-run the training cell and it resumes from the latest HF epoch.
- **Next:** export/quantize the best checkpoint with the repo's
  `scripts/export_onnx.py` -> `scripts/quantize_onnx.py` ->
  `scripts/export_tflite.py` -> `scripts/benchmark_edge.py`.
